# OsteoClassAI v2 · 11 Detector de articulación, paso 2: entrenamiento

Entrena el detector (`osteoclass.detect`: CenterNet pequeño sobre ResNet-18) con las 2889 radiografías de entrenamiento de DetKneeData, elige la mejor época por el IoU del recorte en validación (413) y evalúa una vez en prueba (828).

La métrica principal es el **IoU del recorte**: el cuadrado de lado 1,30 × ancho de la caja que se le daría al clasificador, comparado con el que sale de la caja de Chen.

*Entorno de ejecución:* GPU T4. Luego *Ejecutar todas* (~15 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if os.path.isdir('/content/osteoclassai'):
    !cd /content/osteoclassai && git pull -q
else:
    !git clone -q https://github.com/walterandres23/osteoclassai.git /content/osteoclassai
%cd /content/osteoclassai/v2
!pip -q install -e . h5py opencv-python-headless
!mkdir -p outputs/det && cp -r /content/drive/MyDrive/OsteoClassAI/det/. outputs/det/ && ls outputs/det | head

In [ ]:
!python -m osteoclass.detect --data outputs/det --out runs/detector --epochs 40
!mkdir -p /content/drive/MyDrive/OsteoClassAI/runs/detector && cp -r runs/detector/. /content/drive/MyDrive/OsteoClassAI/runs/detector/

## Revisión visual en prueba: caja de Chen (verde) y recorte detectado (rojo)

In [ ]:
import torch, cv2, pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as pa
from osteoclass.detect import build_model, letterbox, to_tensor, decode, to_original, crop_box
ck = torch.load('runs/detector/best.pt', map_location='cuda')
model = build_model(ck['backbone'], pretrained=False).cuda().eval(); model.load_state_dict(ck['model'])
per = pd.read_csv('runs/detector/pred_test.csv'); idx = pd.read_csv('outputs/det/index.csv')
worst = per.sort_values('crop_iou').image.drop_duplicates().head(3).tolist()
some = per.image.drop_duplicates().sample(3, random_state=0).tolist()
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, path in zip(ax.flat, some + worst):
    gray = cv2.imread(f'outputs/det/{path}', 0); img, s, px, py = letterbox(gray)
    with torch.no_grad(): out = model(to_tensor(img)[None].cuda())[0].float().cpu().numpy()
    a.imshow(gray, cmap='gray'); a.axis('off')
    for _, b in idx[idx.image == path].iterrows():
        x1, y1, x2, y2 = crop_box((b.x1 + b.x2) / 2, (b.y1 + b.y2) / 2, b.x2 - b.x1)
        a.add_patch(pa.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='lime', lw=1.5))
    for k in decode(out[0], out[1:3], out[3]):
        k = to_original(k, s, px, py); x1, y1, x2, y2 = crop_box(k['cx'], k['cy'], k['w'])
        a.add_patch(pa.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='red', lw=1.5, ls='--'))
        a.text(k['cx'], y1 - 3, f"{k['score']:.2f}", color='red', ha='center')
    iou = per[per.image == path].crop_iou.round(2).tolist()
    a.set_title(f"{path.split('/')[-1]}  IoU {iou}" + ('  (peores)' if path in worst else ''))
plt.tight_layout(); plt.show()